In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub", repo_name="llm-zoomcamp",
    commit_id="8c1834d", allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
files = reader.read()
documents = [f.parse() for f in files]

In [2]:
len(documents)

72

In [3]:
from minsearch import Index
index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(documents)

q = "How does the agentic loop keep calling the model until it stops?"
index.search(q, num_results=5)[0]["filename"]

'01-agentic-rag/lessons/14-agentic-loop.md'

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()

In [6]:
from rag_helper import RAGBase

class LessonRAG(RAGBase):

    def search(self, query, num_results=5):
        return self.index.search(query, num_results=num_results)

    def build_context(self, search_results):
        lines = []
        for doc in search_results:
            lines.append(doc['filename'])
            lines.append(doc['content'])
            lines.append('')
        return '\n'.join(lines).strip()

    def llm(self, prompt):
        input_messages = [
            {'role': 'developer', 'content': self.instructions},
            {'role': 'user', 'content': prompt},
        ]
        return self.llm_client.responses.create(
            model=self.model,
            input=input_messages,
        )

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        response = self.llm(prompt)
        return response.output_text, response.usage

In [7]:
rag = LessonRAG(index=index, llm_client=client, model="gpt-5.4-mini")

query = "How does the agentic loop keep calling the model until it stops?"
answer, usage = rag.rag(query)

print(answer)
print("input tokens:", usage.input_tokens)

The loop keeps calling the model with a `while True` loop and stops when the model returns no `function_call` items.

In the code:

- it sends the current `messages` to the model
- if the response contains a function call, it runs the tool and appends the result
- it sets `has_function_calls = True`
- if there are no function calls in that turn, it `break`s out of the loop

So the exit condition is simply: **no tool calls this iteration means the agent is done**.
input tokens: 7111


In [8]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

In [9]:
from minsearch import Index

chunk_index = Index(text_fields=["content"], keyword_fields=["filename"])
chunk_index.fit(chunks)

chunk_rag = LessonRAG(index=chunk_index, llm_client=client, model="gpt-5.4-mini")

query = "How does the agentic loop keep calling the model until it stops?"
answer, usage = chunk_rag.rag(query)
print("chunk input tokens:", usage.input_tokens)

chunk input tokens: 2294


In [11]:
print("Q3 / Q5 ratio:", round(7111 / usage.input_tokens, 1))

Q3 / Q5 ratio: 3.1


In [12]:
search_call_count = 0

def search(query: str) -> list:
    """Search the course lessons for passages relevant to the query.

    Args:
        query: what to look for in the course material
    """
    global search_call_count
    search_call_count += 1
    return chunk_index.search(query, num_results=5)

In [15]:
from toyaikit.tools import Tools
from toyaikit.llm import OpenAIClient
from toyaikit.chat.interface import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner

tools = Tools()
tools.add_tool(search)

instructions = (
    "You're a course teaching assistant. Answer the student's question "
    "using the search tool. Make multiple searches with different keywords "
    "before answering."
)

# wrap your raw OpenAI client — the model name goes HERE, not on the runner
llm_client = OpenAIClient(model="gpt-5.4-mini", client=client)

runner = OpenAIResponsesRunner(
    tools=tools,
    developer_prompt=instructions,        # was 'instructions=' — that's what broke
    chat_interface=IPythonChatInterface(),
    llm_client=llm_client,                # wrapper, not the raw client
)

In [16]:
search_call_count = 0

question = "How does the agentic loop work, and how is it different from plain RAG?"
result = runner.loop(prompt=question, callback=runner.displaying_callback)

print("search was called:", search_call_count, "times")

-> Response received


-> Response received


search was called: 4 times
